<a href="https://colab.research.google.com/github/melissa-04/melisayla-biyoinformatik/blob/main/notebooks/rna-seq/11_sanik.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sanık mı, kurban mı?

Bu rehber bir hatanın düzeltmesidir ve hata benim: sekizinci rehberde Ahsp'yi "nakavt edilen gen" ilan ettim. Rehber 1 baştan beri başka bir şey söylüyordu — bu deney bir **Klf1** nakavtı. Bu defterde iki geni yan yana koyup kanıta bakacağız ve daha önemlisi, hatanın nereden çıktığını söküp bir kurala çevireceğiz. Kurulum yok, defter dakikada biter.

In [1]:
import pandas as pd
import numpy as np

t = pd.read_csv('https://raw.githubusercontent.com/melissa-04/melisayla-biyoinformatik/main/data/rna-seq/sayim_tablosu.csv')
ornekler = ['WT_1', 'WT_2', 'WT_3', 'KO_1', 'KO_2', 'KO_3']
g = t.set_index('gen_id')[ornekler]

poz = g[(g > 0).all(axis=1)]
logg = np.log(poz)
sf = np.exp(logg.sub(logg.mean(axis=1), axis=0).median(axis=0))
norm = g.div(sf, axis=1)

def gen(ad, tablo):
    return tablo.loc[t.loc[t.gen_adi == ad, 'gen_id'].iloc[0]]

print('hazır')

hazır


## 1. İki satır yan yana

Önce iki şüpheli, normalize değerleriyle.

In [2]:
for ad in ['Klf1', 'Ahsp']:
    v = gen(ad, norm)
    wt, ko = v[:3].mean(), v[3:].mean()
    print(f'{ad:5s}  WT ort: {wt:8.0f}   KO ort: {ko:6.0f}   KO/WT: {ko/wt:.4f}')

Klf1   WT ort:    11449   KO ort:   1296   KO/WT: 0.1132
Ahsp   WT ort:    99368   KO ort:     13   KO/WT: 0.0001


## 2. Sessiz düşüş, mutlak susuş

İki desen birbirinden farklı. Klf1 sekizde birine düşmüş ama susmamış: KO örneklerinde bin küsur sayımlık bir kalıntı var — nakavt kasetlerinin geride bıraktığı işlevsiz transkript ya da arka plan; RNA-seq molekülü sayar, işlevi ölçmez. Ahsp ise mutlak susmuş: on binlerden tek hanelere. Bir transkripsiyon faktörü gittiğinde kendi geni "azalır", ama tamamen ona bağımlı bir hedefin promotörü "kapanır". Yedinci rehberdeki av kriterimiz — en küçük KO/WT oranı — bu yüzden nakavtı değil, en bağımlı kurbanı buldu.

Peki nakavtın kendisi o gün neredeydi? Klf1'in oranı 0,11: aramayı "en uç" yerine "tanıdık" üzerinden yapsaydık bile 8. rehberin volkanında oradaydı, lfc −3,1 ve ezici bir padj ile — sadece kimse ona bakmadı, çünkü kriter başka şey arıyordu.

## 3. Hedef ağı

Tek kurban hikâye kurmaz; ağ kurar. KLF1'in bilinen hedeflerinden bir panel:

In [3]:
panel = ['Klf1', 'Ahsp', 'Dmtn', 'Hbb-bs', 'E2f2', 'Slc4a1', 'Epb42', 'Hbb-y']
satirlar = []
for ad in panel:
    v = gen(ad, norm)
    wt, ko = v[:3].mean(), v[3:].mean()
    satirlar.append((ad, round(wt), round(ko), round(ko / wt, 4)))
print(pd.DataFrame(satirlar, columns=['gen', 'WT_ort', 'KO_ort', 'KO/WT']).to_string(index=False))

   gen  WT_ort  KO_ort  KO/WT
  Klf1   11449    1296 0.1132
  Ahsp   99368      13 0.0001
  Dmtn   10546      61 0.0058
Hbb-bs   37039     570 0.0154
  E2f2    9443     990 0.1049
Slc4a1  107508   36082 0.3356
 Epb42    5552    1861 0.3352
 Hbb-y   13956  131603 9.4299


## 4. Okuma

Zar iskeleti (Dmtn, Slc4a1, Epb42), yetişkin globin (Hbb-bs), hücre döngüsü sürücüsü (E2f2), şaperon (Ahsp): hepsi KLF1 hedefi, hepsi aşağıda — ve embriyonik Hbb-y dokuz kat yukarıda, çünkü KLF1 embriyonik→yetişkin globin anahtarlamasının da düğmesidir. Dokuzuncu rehberin zenginleştirme tablosu, onuncu rehberin ısı haritası, sekizin volkanı: hepsi aynı ağın çöküşünü değişik açılardan çekmiş. Hbb-bs bilmecesi de son hâlini alıyor: düşüşün sağlam bir biyolojik bacağı var (KLF1 hedefi), arındırma belirsizliği o bacağın üstüne biner — yön güvenilir, büyüklük değil.

Üç kural çıkarıyorum. Bir: kriter neyi ararsa onu bulur; "en çok çöken" sorusunun cevabı nakavt değildir, tanım gereği en bağımlı hedeftir. İki: deneyin tasarım belgesi — bizde Rehber 1 ve Veri sayfası — analizin süsü değil parçasıdır; sonuç, tasarımla çapraz kontrol edilmeden ilan edilmez. Üç: yanlış ilan edilmişse düzeltme görünür yapılır; sekizinci rehberdeki düzeltme kutusu bilerek orada duracak.

## Kendin dene

İki kısa görev. Birincisi: kendi koşunuzdan Klf1 ve Ahsp'nin KO/WT oranlarını yazın ve tek cümleyle bitirin: sanık hangisi, kanıtınız ne? İkincisi: bir sonraki projenizde elinize bir DE listesi geçtiğinde "nakavt edilen gen hangisi" sorusuna nasıl yaklaşacağınızı tek cümleyle yazın — bu cümle sizin kuralınız olacak.